# Un Algoritmo di Correzione per un Motore di Ricerca

## Caso d'Uso Aziendale: Searchify

### Introduzione all'Azienda
Searchify è una startup innovativa che offre un motore di ricerca personalizzato per aziende di medie dimensioni, consentendo di cercare informazioni specifiche nei loro database interni. Uno dei principali vantaggi di Searchify è la semplicità d'uso: gli utenti possono trovare rapidamente documenti, report e altre risorse aziendali digitando parole chiave. Tuttavia, l'azienda si trova ad affrontare un problema che mina l'esperienza dell'utente.

### Problema
Il problema principale di Searchify è che molti utenti commettono errori di digitazione durante l'inserimento delle parole chiave. Questi errori causano risultati di ricerca nulli o non pertinenti, portando a insoddisfazione tra gli utenti. Ad esempio, se un utente cerca "raporto vendite 2023" invece di "rapporto vendite 2023", il motore di ricerca non restituisce alcun risultato.

### Obiettivo del Progetto
L'obiettivo è sviluppare un algoritmo di correzione automatica per il motore di ricerca di Searchify. Questo algoritmo dovrà:

1. Rilevare automaticamente gli errori di digitazione o le parole non valide.
2. Suggerire la parola corretta più probabile.
3. Restituire risultati pertinenti basati sulla correzione suggerita.

L'implementazione di questa funzionalità migliorerà notevolmente l'esperienza utente, aumentando l'efficienza del motore di ricerca e la soddisfazione dei clienti.

### Benefici Attesi
- Miglioramento dell'accuratezza: Riduzione degli errori di ricerca dovuti a digitazioni errate.
- Incremento della produttività: Gli utenti troveranno le informazioni più velocemente.
- Aumento della fedeltà dei clienti: Un'esperienza utente più fluida e soddisfacente porterà a una maggiore adozione del prodotto.

### Specifiche del Progetto
1. Input: Una stringa rappresentante una query di ricerca, digitata dall'utente.
2. Output:
    - Se la query contiene errori, il sistema suggerisce una correzione.
    - Se la query è corretta, restituisce la stessa query.
3. Funzionalità Chiave:
    - Confronto tra la parola inserita e un dizionario di parole corrette (da predefinire nel codice).
    - Implementazione di un algoritmo che calcoli la "distanza di edit" (es. distanza di Levenshtein) per trovare le parole più simili.
    - Gestione di casi d'uso realistici come errori di battitura comuni, lettere scambiate, omissioni o aggiunte.

### Consegna
Scrivi un programma Python che implementi l'algoritmo di correzione automatica per il motore di ricerca. Il tuo codice deve:

1. Contenere una funzione principale chiamata suggest_correction(query, dictionary).
    - Parametri:
        - query: La stringa di input inserita dall'utente.
        - dictionary: Una lista di parole corrette.
    - Ritorno:
        - La parola corretta più probabile o la query originale se è già valida.
2. Testare il funzionamento con almeno 10 casi d'uso (inclusi errori comuni e query corrette).
3. Essere ben documentato, con commenti che spieghino le principali scelte implementative.
4. Fornire un dizionario base contenente almeno 50 parole.

### Nota
Il progetto deve essere completato senza l'uso di librerie esterne per il calcolo della distanza di edit o per altre funzionalità. L'obiettivo è che lo studente implementi l'algoritmo interamente da zero.

## Soluzione Proposta
Algoritmo di Correzione Automatica per il Motore di Ricerca Searchify

![levensthein_distance](levensthein_distance.webp)

In [ ]:
def levenshtein_distance(word1, word2):
    """
    Calcola la distanza di Levenshtein tra due parole.
    
    La distanza di Levenshtein è il numero minimo di operazioni (inserimento,
    cancellazione, sostituzione) necessarie per trasformare una parola nell'altra.
    
    Args:
        word1 (str): Prima parola
        word2 (str): Seconda parola
        
    Returns:
        int: La distanza di Levenshtein tra le due parole
    """

    if type(word1) != str or type(word2) != str:
        raise ValueError("Entrambi gli argomenti devono essere stringhe.")
    if not word1 or not word2:
        raise ValueError("Le parole non possono essere vuote.")

    len1 = len(word1)
    len2 = len(word2)

    # Crea una matrice (len1+1) x (len2+1) inizializzando tutti i valori a 0
    matrix = [[0] * (len2 + 1) for _ in range(len1 + 1)]

    # Inizializza la prima riga e colonna
    for i in range(len1 + 1):
        matrix[i][0] = i
    for j in range(len2 + 1):
        matrix[0][j] = j

    # Riempie la matrice
    for i in range(1, len1 + 1):
        for j in range(1, len2 + 1):
            if word1[i - 1] == word2[j - 1]:
                matrix[i][j] = matrix[i - 1][j - 1]
            else:
                # Prende il minimo tra:
                # - Sostituzione (diagonale): matrix[i-1][j-1] + 1
                # - Cancellazione (sopra): matrix[i-1][j] + 1
                # - Inserimento (sinistra): matrix[i][j-1] + 1
                matrix[i][j] = 1 + min(matrix[i - 1][j - 1],
                                       matrix[i - 1][j],
                                       matrix[i][j - 1])

    return matrix[len1][len2]


def suggest_correction(query, dictionary, lowercase=False):
    """
    Suggerisce una correzione per una parola digitata in modo errato.
    
    Confronta la query con tutte le parole nel dizionario e suggerisce
    quella con la minore distanza di Levenshtein.
    
    Args:
        query (str): La parola digitata dall'utente
        dictionary (list): Lista di parole corrette
        lowercase (bool): Se True, ignora le differenze tra maiuscole e minuscole
        
    Returns:
        str: La parola corretta suggerita o la query originale se è già corretta
    """
    if type(query) != str:
        raise ValueError("La query deve essere una stringa.")
    if not query:
        raise ValueError("La query non può essere vuota.")

    distances = []

    if lowercase:
        # Se la query è già nel dizionario, ritorna la query originale
        if query.lower() in [word.lower() for word in dictionary]:
            return query

        # Calcola la distanza per ogni parola nel dizionario
        for word in dictionary:
            distance = levenshtein_distance(query.lower(), word.lower())
            distances.append((word, distance))

    else:
        # Se la query è già nel dizionario, ritorna la query originale
        if query in dictionary:
            return query

        # Calcola la distanza per ogni parola nel dizionario
        for word in dictionary:
            distance = levenshtein_distance(query, word)
            distances.append((word, distance))

    # Ordina per distanza (la parola più simile sarà la prima)
    distances.sort(key=lambda x: x[1])

    # Ritorna la parola con la minore distanza
    suggested_word = distances[0][0]
    min_distance = distances[0][1]

    # Se la distanza è troppo grande (> 2), potrebbe non essere una correzione affidabile
    # Ma comunque la suggeriamo
    return suggested_word


## Database

In [ ]:
# Dizionario base di parole corrette (almeno 50 parole)
dictionary = [
    # Parole comuni di business
    "rapporto", "vendite", "cliente", "prodotto", "servizio", "azienda",
    "database", "software", "hardware", "rete", "server", "documento",
    "progetto", "budget", "fattura", "ricevuta", "contratto", "accordo",
    "strategia", "marketing", "pubblicità", "analisi", "dati", "statistiche",

    # Parole comuni
    "ricerca", "risultato", "errore", "correzione", "motore", "query",
    "parola", "lettera", "digitazione", "modello", "algoritmo", "funzione",
    "variabile", "lista", "dizionario", "stringa", "numero", "intero",
    "profilo", "account", "password", "utente", "nome", "cognome",
    "indirizzo", "città", "paese", "telefono", "email", "messaggio",

    # Azioni comuni
    "aggiorna", "crea", "modifica", "elimina", "salva", "carica",
    "esporta", "importa", "scarica", "condividi", "invia", "ricevi"
]

print(f"Dizionario caricato con {len(dictionary)} parole")
print(f"Prime 10 parole: {dictionary[:10]}")


## Test del codice

In [ ]:
# Test cases: almeno 10 casi d'uso per verificare il funzionamento

test_cases = [
    # (query, expected_result_description)
    ("raparto", "raporto → rapporto (errore comune: lettera scambiata)"),
    ("rendite", "vendite"),
    ("clinte", "cliente"),
    ("prodotto", "prodotto (parola già corretta)"),
    ("servizio", "servizio (parola già corretta)"),
    ("agienda", "azienda (errore di digitazione)"),
    ("databse", "database (errore comune: lettere scambiate)"),
    ("sofware", "software (omissione di lettera)"),
    ("analissi", "analisi (lettera doppia errata)"),
    ("modelo", "modello (omissione di lettera)"),
    ("algorimo", "algoritmo (omissione di lettera)"),
    ("utne", "utente (errore di digitazione)"),
    ("messaggio", "messaggio (parola già corretta)"),
]

print("=" * 80)
print("TEST DEL SISTEMA DI CORREZIONE AUTOMATICA")
print("=" * 80)
print()

for i, (query, description) in enumerate(test_cases, 1):
    suggestion = suggest_correction(query, dictionary)
    distance = levenshtein_distance(query, suggestion)

    print(f"Test {i}: INPUT: '{query}'")
    print(f"  → OUTPUT: '{suggestion}'")
    print(f"  → DISTANZA: {distance}")
    print(f"  → DESCRIZIONE: {description}")
    print()


In [ ]:
# Analisi dettagliata della distanza di Levenshtein
print("=" * 80)
print("ANALISI DETTAGLIATA: Come funziona l'algoritmo")
print("=" * 80)
print()

# Esempio: "raparto" → "rapporto"
word1 = "raparto"
word2 = "rapporto"
print(f"Confronto tra '{word1}' e '{word2}':")
print(f"Distanza di Levenshtein: {levenshtein_distance(word1, word2)}")
print(f"Spiegazione: Sono necessari 2 cambiamenti:")
print(f"  1. 'raparto' → 'rapporto' (cambio di 'a' con 'o' e inserimento di 'p')")
print()

# Esempio: "rendite" → "vendite"
word1 = "rendite"
word2 = "vendite"
print(f"Confronto tra '{word1}' e '{word2}':")
print(f"Distanza di Levenshtein: {levenshtein_distance(word1, word2)}")
print(f"Spiegazione: È necessario 1 cambiamento:")
print(f"  1. 'r' → 'v' all'inizio della parola")
print()


In [ ]:
# Funzione helper per visualizzare i risultati di correzione
def show_correction_details(query, dictionary):
    """
    Mostra dettagli della correzione con le 5 parole più simili.
    """
    query_lower = query.lower()

    # Se la parola è corretta, lo segnala
    if query_lower in [word.lower() for word in dictionary]:
        print(f"✓ La parola '{query}' è corretta!")
        return

    # Calcola le distanze per tutte le parole
    distances = []
    for word in dictionary:
        distance = levenshtein_distance(query, word)
        distances.append((word, distance))

    # Ordina per distanza
    distances.sort(key=lambda x: x[1])

    # Mostra i risultati
    print(f"Input: '{query}'")
    print(f"Suggerimento principale: '{distances[0][0]}' (distanza: {distances[0][1]})")
    print(f"\nTop 5 candidati:")
    for i, (word, distance) in enumerate(distances[:5], 1):
        print(f"  {i}. '{word}' - distanza {distance}")
    print()


# Test con alcuni esempi interattivi
print("=" * 80)
print("ESEMPI INTERATTIVI DI CORREZIONE")
print("=" * 80)
print()

# Esempi con parole errate
examples = ["raparto", "databse", "clinte", "algoritmo"]
for example in examples:
    show_correction_details(example, dictionary)
